# Fixing Links Notebook

Before running this notebook, make sure to run the following command in the terminal to install the required packages:

```bash
bundle install
make all
ruby parse_htmlproofer_log.rb 
```

Each command should be run separately and the final two commands create files for all the htmlproofer errors and warnings. This notebook loads the final csv file to help you see what links exists. You will also need to install the `pandas` library if you haven't already. You can do this by running:

```bash
pip install pandas
```

## Load Libraries and Data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("htmlproofer-report.csv")
# Lower case the column names
df.columns = df.columns.str.lower()
print(f"Number of errors: {len(df)}")

Number of errors: 74


In [4]:
message_counts = df.message.value_counts().reset_index()
print(f"Number of unique error messages: {len(message_counts)}")
message_counts[(message_counts['count']>1)]

Number of unique error messages: 38


,message,count
0,External link https://twitter.com/jenniferisve...,8
1,External link https://central.github.com/mac/l...,4
2,External link https://twitter.com/alexwermerco...,4
3,External link https://twitter.com/araceletorre...,4
4,External link https://twitter.com/Adam_Crymble...,4
5,http://www.geonames.org/ is not an HTTPS link,3
6,http://mouapp.com/ is not an HTTPS link,3
7,http://www.sublimetext.com/ is not an HTTPS link,3
8,http://prose.io/ is not an HTTPS link,2
9,http://library.gwu.edu/scholarly-technology-gr...,2


In [5]:
external_links = df[df['message'].str.contains("External link")].copy()
internal_links = df[df['message'].str.contains("internally linking")].copy()
print(f"Number of external link errors: {len(external_links)}")
print(f"Number of internal link errors: {len(internal_links)}")

Number of external link errors: 33
Number of internal link errors: 0


In [6]:
file_counts = df.file.value_counts().reset_index()
print(f"Number of unique files with errors: {len(file_counts)}")
file_counts[file_counts['count']>1]

Number of unique files with errors: 29


,file,count
0,_site/es/lecciones/escritura-sostenible-usando...,6
1,_site/fr/lecons/intro-donnees-ouvertes-liees/i...,6
2,_site/fr/lecons/redaction-durable-avec-pandoc-...,6
3,_site/en/project-team/index.html,5
4,_site/es/equipo-de-proyecto/index.html,5
5,_site/fr/equipe-projet/index.html,5
6,_site/pt/equipe/index.html,5
7,_site/fr/lecons/publier-archives-tei-ceteicean...,4
8,_site/en/lessons/retired/intro-to-augmented-re...,4
9,_site/en/lessons/retired/getting-started-with-...,3


In [7]:
file_counts_df = df.file.value_counts().reset_index()
file_counts_df['count_index'] = file_counts_df.index

file_counts_df

,file,count,count_index
0,_site/es/lecciones/escritura-sostenible-usando...,6,0
1,_site/fr/lecons/intro-donnees-ouvertes-liees/i...,6,1
2,_site/fr/lecons/redaction-durable-avec-pandoc-...,6,2
3,_site/en/project-team/index.html,5,3
4,_site/es/equipo-de-proyecto/index.html,5,4
5,_site/fr/equipe-projet/index.html,5,5
6,_site/pt/equipe/index.html,5,6
7,_site/fr/lecons/publier-archives-tei-ceteicean...,4,7
8,_site/en/lessons/retired/intro-to-augmented-re...,4,8
9,_site/en/lessons/retired/getting-started-with-...,3,9


In [8]:
merged_df = df.merge(file_counts_df, on='file', how='outer').sort_values(by="count_index", ascending=True)

In [10]:
# import os
# import re

# EXTENSIONS = (".yml")

# def replace_links_preserving_code_blocks(file_path):
#     with open(file_path, "r", encoding="utf-8") as f:
#         content = f.read()

#     # Match code blocks (triple backticks) and inline code (`...`)
#     code_blocks = list(re.finditer(r"(```.*?```|`[^`]*`)", content, re.DOTALL))
#     modified = content
#     offset = 0

#     for match in code_blocks:
#         start, end = match.span()
#         segment = content[start:end]

#         # Temporarily mark this section to skip
#         placeholder = f"%%CODEBLOCK{start}%%"
#         modified = modified[:start + offset] + placeholder + modified[end + offset:]
#         offset += len(placeholder) - (end - start)

#     # Replace all http:// with https://
#     modified = re.sub(r"http://", "https://", modified)

#     # Restore code blocks untouched
#     for match in code_blocks:
#         start = match.start()
#         placeholder = f"%%CODEBLOCK{start}%%"
#         modified = modified.replace(placeholder, match.group(0))

#     if content != modified:
#         print(f"✅ Updated: {file_path}")
#         with open(file_path, "w", encoding="utf-8") as f:
#             f.write(modified)

# def process_all_files(root="."):
#     for dirpath, _, filenames in os.walk(root):
#         for fname in filenames:
#             if fname.endswith(EXTENSIONS) and "ph_authors" in fname:
#                 replace_links_preserving_code_blocks(os.path.join(dirpath, fname))

# process_all_files()